In [1]:
import pandas as pd
df = pd.read_parquet("../classify_data.parquet").reset_index()

In [2]:
class_map = {
    '0': 0,
    '1-3': 1,
    '4-7': 2,
    '8-14': 3,
    '15-30': 4,
    '31-60': 5
}
df['resolution_class_int'] = df['resolution_class'].map(class_map)

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

feature_columns = ['Agency','Complaint Type','Descriptor','Location Type', 'Incident Zip', 'City', 'Borough',
                    'Open Data Channel Type', 'Latitude', 'Longitude',
                    'hour_of_day', 'day_of_week', 'month', 'is_weekend', 'open_requests_past_24h']

X = df[feature_columns]
y = df['resolution_class_int']

#Segregate columsn for preprocessing
categorical_cols = ['Agency','Complaint Type','Descriptor','Location Type', 'Incident Zip', 'City', 'Borough',
                    'Open Data Channel Type' ,'hour_of_day','day_of_week', 'month', 'is_weekend']
numeric_cols = ['Latitude','Longitude','open_requests_past_24h']

# Preprocessing -> Normalization + encoding categories
preprocess = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', StandardScaler(), numeric_cols)
    ]
)

lr = LogisticRegression(
    solver='saga',     
    max_iter=200,      
    tol=0.005,
    class_weight='balanced',
    n_jobs=8,         
    verbose=3         
)

#Full pipeline: preprocessing + model
model = Pipeline(steps=[
    ('preprocess', preprocess),
    ('clf', lr)
])

#Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [5]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
scores = cross_val_score(model, X_train, y_train, cv=skf, scoring='f1_macro')

print(scores)

# 6. Fit model
model.fit(X_train, y_train)

# 7. Predict
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)

[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.


Epoch 1, change: 1
Epoch 2, change: 0.10484029
Epoch 3, change: 0.083906576
Epoch 4, change: 0.071569991
Epoch 5, change: 0.03451932
Epoch 6, change: 0.029408192
Epoch 7, change: 0.023283493
Epoch 8, change: 0.020492808
Epoch 9, change: 0.018152347
Epoch 10, change: 0.015926568
Epoch 11, change: 0.014375769
Epoch 12, change: 0.013842243
Epoch 13, change: 0.013854393
Epoch 14, change: 0.011588027
Epoch 15, change: 0.010876178
Epoch 16, change: 0.01014558
Epoch 17, change: 0.0095173808
Epoch 18, change: 0.0090553346
Epoch 19, change: 0.0083691284
Epoch 20, change: 0.0081452744
Epoch 21, change: 0.0075658117
Epoch 22, change: 0.0071856677
Epoch 23, change: 0.0069339406
Epoch 24, change: 0.006557766
Epoch 25, change: 0.0064155916
Epoch 26, change: 0.0062953023
Epoch 27, change: 0.0062160423
Epoch 28, change: 0.0061261252
Epoch 29, change: 0.0060376771
Epoch 30, change: 0.0053496447
Epoch 31, change: 0.0052603389
Epoch 32, change: 0.0051910626
Epoch 33, change: 0.0051196956
convergence afte

[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.


Epoch 1, change: 1
Epoch 2, change: 0.095250631
Epoch 3, change: 0.086238159
Epoch 4, change: 0.048060411
Epoch 5, change: 0.037892068
Epoch 6, change: 0.036444909
Epoch 7, change: 0.0262108
Epoch 8, change: 0.023076359
Epoch 9, change: 0.020176578
Epoch 10, change: 0.018382654
Epoch 11, change: 0.016435832
Epoch 12, change: 0.015399753
Epoch 13, change: 0.014849116
Epoch 14, change: 0.014317436
Epoch 15, change: 0.013507258
Epoch 16, change: 0.013138067
Epoch 17, change: 0.01221469
Epoch 18, change: 0.012041824
Epoch 19, change: 0.011686884
Epoch 20, change: 0.010730073
Epoch 21, change: 0.010488665
Epoch 22, change: 0.0095424773
Epoch 23, change: 0.0093808256
Epoch 24, change: 0.0089133953
Epoch 25, change: 0.0088313958
Epoch 26, change: 0.0079631983
Epoch 27, change: 0.0078426548
Epoch 28, change: 0.0067761509
Epoch 29, change: 0.0068983118
Epoch 30, change: 0.0062523164
Epoch 31, change: 0.0063187369
Epoch 32, change: 0.005810821
Epoch 33, change: 0.0052985365
Epoch 34, change: 0.0

[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.


Epoch 1, change: 1
Epoch 2, change: 0.10474181
Epoch 3, change: 0.053375859
Epoch 4, change: 0.069956695
Epoch 5, change: 0.063334124
Epoch 6, change: 0.029954452
Epoch 7, change: 0.019369897
Epoch 8, change: 0.02147731
Epoch 9, change: 0.014336738
Epoch 10, change: 0.014017568
Epoch 11, change: 0.013355556
Epoch 12, change: 0.013126386
Epoch 13, change: 0.012745203
Epoch 14, change: 0.01227148
Epoch 15, change: 0.011807038
Epoch 16, change: 0.011473606
Epoch 17, change: 0.010882965
Epoch 18, change: 0.01051731
Epoch 19, change: 0.010108104
Epoch 20, change: 0.0093788724
Epoch 21, change: 0.0089866764
Epoch 22, change: 0.0088941618
Epoch 23, change: 0.0083136872
Epoch 24, change: 0.0080448406
Epoch 25, change: 0.007719982
Epoch 26, change: 0.0068997993
Epoch 27, change: 0.0066827476
Epoch 28, change: 0.006311868
Epoch 29, change: 0.0061319471
Epoch 30, change: 0.0056675636
Epoch 31, change: 0.0055134728
Epoch 32, change: 0.0052043327
convergence after 33 epochs took 40 seconds
[0.43633

[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.


Epoch 1, change: 1
Epoch 2, change: 0.11519834
Epoch 3, change: 0.062941196
Epoch 4, change: 0.043475876
Epoch 5, change: 0.036059266
Epoch 6, change: 0.028159311
Epoch 7, change: 0.022261948
Epoch 8, change: 0.017728195
Epoch 9, change: 0.016783564
Epoch 10, change: 0.015675842
Epoch 11, change: 0.01504429
Epoch 12, change: 0.014257219
Epoch 13, change: 0.013318711
Epoch 14, change: 0.011726526
Epoch 15, change: 0.011291824
Epoch 16, change: 0.010669988
Epoch 17, change: 0.0099435866
Epoch 18, change: 0.0094433817
Epoch 19, change: 0.0090826941
Epoch 20, change: 0.0085572669
Epoch 21, change: 0.0081516181
Epoch 22, change: 0.0075858366
Epoch 23, change: 0.0072299879
Epoch 24, change: 0.0067384035
Epoch 25, change: 0.0064012658
Epoch 26, change: 0.0062572035
Epoch 27, change: 0.0061397514
Epoch 28, change: 0.0056218853
Epoch 29, change: 0.005688197
Epoch 30, change: 0.0053403563
Epoch 31, change: 0.0052422626
Epoch 32, change: 0.0051449993
Epoch 33, change: 0.0050630002
convergence aft

In [6]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Test Accuracy: 0.7276902639662405

Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.84      0.90    331345
           1       0.52      0.57      0.55     76042
           2       0.26      0.34      0.30     27143
           3       0.23      0.28      0.25     17086
           4       0.20      0.36      0.26     15160
           5       0.24      0.61      0.35     10482

    accuracy                           0.73    477258
   macro avg       0.41      0.50      0.43    477258
weighted avg       0.80      0.73      0.76    477258


Confusion Matrix:
 [[277575  33533   7972   3444   4472   4349]
 [  4619  43694  12330   5001   5976   4422]
 [   570   4678   9249   4320   4884   3442]
 [   251   1131   3167   4863   4488   3186]
 [   130    544   1822   2645   5520   4499]
 [    68    134    568    826   2491   6395]]


In [6]:
import numpy as np
np.save("../results/regresssion_predictions.npy", y_pred)
np.save("../results/regresssion_prob.npy", y_prob)

In [ ]:
y_pred = np.load("../results/regresssion_predictions.npy")

NameError: name 'np' is not defined

In [ ]:
print(X.shape)
print(y.shape)
df.dtypes